# Yuda 4 — VLM Scale to Full Dataset
4 experiments: Zero-shot (A), Conf Filter (B), Text Features (C), Multi-Encoder (D)

In [1]:
import sys
sys.path.insert(0, "../..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader

import open_clip
import timm

from modules.utils.config import *
from modules.utils.seed import set_seed
from modules.utils.load_data import load_train
from modules.models.factory import TrashClassifier
from modules.training.train import fit
from modules.data.dataset import get_dataloaders, get_transforms

set_seed(SEED)
print(f"Device: {DEVICE}")
print(f"open_clip: {open_clip.__version__}")

/home/prayudahlah/projects/external/big-data-big-trouble/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
open_clip: 3.3.0


In [2]:
from sklearn.model_selection import StratifiedShuffleSplit
_CLASS_TO_IDX = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}

# CLIP model & transforms
CLIP_NAME = 'ViT-B-32'
CLIP_PRETRAINED = 'laion2b_s34b_b79k'
clip_model, _, clip_transform = open_clip.create_model_and_transforms(
    CLIP_NAME, pretrained=CLIP_PRETRAINED
)
clip_tokenizer = open_clip.get_tokenizer(CLIP_NAME)
clip_model = clip_model.to('cpu')
clip_model.eval()
clip_visual = clip_model.visual.to(DEVICE)
clip_dim = clip_visual.output_dim
print(f"CLIP visual dim: {clip_dim}")

# ConvNeXt transforms
conv_train_tfm = get_transforms(augment=True, img_size=224)
conv_val_tfm = get_transforms(augment=False, img_size=224)

results = []

CLIP visual dim: 512


In [3]:
# Full dataset dataloaders
train_loader, val_loader, val_ds = get_dataloaders(batch_size=32, num_workers=4)
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")

# CLIP val loader
class CLIPValDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row['path']).convert('RGB')
        return self.transform(img), _CLASS_TO_IDX[row['label']]

df_train = train_loader.dataset.df
df_val = val_loader.dataset.df
clip_val_ds = CLIPValDataset(df_val, clip_transform)
clip_val_loader = DataLoader(clip_val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
print(f"CLIP Val loader: {len(clip_val_loader)} batches")

Train: 21221 | Val: 5306
CLIP Val loader: 166 batches


---
## A: Zero-Shot Baseline

In [4]:
print("=" * 50)
print("A: Zero-Shot")
print("=" * 50)

prompts = ["recyclable waste", "electronic waste", "organic waste"]
text_tokens = clip_tokenizer(prompts).to('cpu')
with torch.no_grad():
    text_feat = clip_model.encode_text(text_tokens)
    text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
text_feat = text_feat.to(DEVICE)

all_preds, all_labels, all_conf = [], [], []
for images, labels in tqdm(clip_val_loader, desc="Zero-shot"):
    images = images.to(DEVICE)
    with torch.no_grad():
        img_feat = clip_visual(images)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        sim = (100.0 * img_feat @ text_feat.T).softmax(dim=-1)
        conf, preds = sim.max(dim=1)
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(labels.numpy())
    all_conf.extend(conf.cpu().numpy())

f1 = f1_score(all_labels, all_preds, average='macro')
f1_per = f1_score(all_labels, all_preds, average=None)
all_conf = np.array(all_conf)
print(f"Macro F1: {f1:.4f}")
print(f"Per-class F1: {f1_per}")
print(f"Confidence: mean={all_conf.mean():.4f} median={np.median(all_conf):.4f}")
for th in [0.90, 0.95, 0.99]:
    pct = (all_conf >= th).mean() * 100
    print(f"  >= {th}: {pct:.1f}%")
results.append({'name': 'A_zero_shot', 'best_val_f1': f1, 'f1_per_class': f1_per.tolist()})

A: Zero-Shot


Zero-shot: 100%|██████████| 166/166 [00:11<00:00, 14.10it/s]

Macro F1: 0.8520
Per-class F1: [0.79957356 0.86507937 0.89128728]
Confidence: mean=0.8663 median=0.9380
  >= 0.9: 58.8%
  >= 0.95: 46.6%
  >= 0.99: 24.2%


## B: Confidence Filter
CLIP ≥ threshold → langsung predict, sisanya ConvNeXt

In [5]:
print("=" * 50)
print("B: Confidence Filter")
print("=" * 50)

# Save CLIP preds from Cell 4 (still in memory)
clip_save_preds = np.array(all_preds)
clip_save_conf = np.array(all_conf)
clip_save_labels = np.array(all_labels)

# Free CLIP from GPU
import gc
del clip_visual, text_feat
gc.collect()
torch.cuda.empty_cache()

# Load ConvNeXt alone (with correct transforms)
from modules.data.dataset import TrashDataset as TrashDatasetCls
conv_val_ds = TrashDatasetCls(df_val, transform=conv_val_tfm)
conv_val_loader = DataLoader(conv_val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

convnext = TrashClassifier('convnext_tiny', num_classes=3).to(DEVICE)
ckpt = torch.load(RESULTS / 'yuda_convnext_tiny.pt', map_location=DEVICE, weights_only=True)
convnext.load_state_dict(ckpt)
convnext.eval()

# Run ConvNeXt on validation set
all_conv_preds = []
for images, _ in tqdm(conv_val_loader, desc="ConvNeXt on val"):
    images = images.to(DEVICE)
    with torch.no_grad():
        preds = convnext(images).argmax(dim=1).cpu().numpy()
    all_conv_preds.extend(preds)
all_conv_preds = np.array(all_conv_preds)

del convnext
gc.collect()
torch.cuda.empty_cache()

# Combine with numpy (no GPU needed)
for TRESHOLD in [0.90, 0.95, 0.99]:
    high_conf = clip_save_conf >= TRESHOLD
    final_preds = np.where(high_conf, clip_save_preds, all_conv_preds)
    f1 = f1_score(clip_save_labels, final_preds, average='macro')
    f1_per = f1_score(clip_save_labels, final_preds, average=None)
    print(f"Threshold {TRESHOLD}: CLIP confident={high_conf.sum()}/{len(high_conf)} | F1={f1:.4f} | per-class={f1_per}")
    results.append({'name': f'B_conf_filter_{TRESHOLD}', 'best_val_f1': f1, 'f1_per_class': f1_per.tolist()})

B: Confidence Filter


ConvNeXt on val: 100%|██████████| 166/166 [00:18<00:00,  9.09it/s]

Threshold 0.9: CLIP confident=3119/5306 | F1=0.9491 | per-class=[0.93527013 0.95051924 0.96139378]
Threshold 0.95: CLIP confident=2474/5306 | F1=0.9551 | per-class=[0.9426687  0.95828221 0.96428571]
Threshold 0.99: CLIP confident=1285/5306 | F1=0.9571 | per-class=[0.94473883 0.96232242 0.96421471]


## C: Text Embedding Features
CLIP visual features + text similarity → classifier

In [6]:
# Reload CLIP visual (freed by Cell 5)
clip_visual = clip_model.visual.to(DEVICE)

class CLIPWithTextFeatures(nn.Module):
    def __init__(self, clip_model, clip_visual, num_classes=3):
        super().__init__()
        self.visual = clip_visual
        self.encoder = self.visual
        self.logit_scale = clip_model.logit_scale
        dim = self.visual.output_dim
        prompts = ["recyclable waste", "electronic waste", "organic waste"]
        text_tokens = clip_tokenizer(prompts)
        with torch.no_grad():
            text_feat = clip_model.encode_text(text_tokens)
            text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        self.register_buffer('text_features', text_feat)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(dim + 3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        v = self.visual(x)
        v_norm = v / v.norm(dim=-1, keepdim=True)
        sim = v_norm @ self.text_features.T
        combined = torch.cat([v, sim * self.logit_scale.exp()], dim=1)
        return self.classifier(combined)
    def freeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = True


class CLIPTrainDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row['path']).convert('RGB')
        return self.transform(img), _CLASS_TO_IDX[row['label']]

# Build CLIP train/val loaders
clip_train_ds = CLIPTrainDataset(df_train, clip_transform)
clip_val_ds = CLIPTrainDataset(df_val, clip_transform)
clip_train_loader = DataLoader(clip_train_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
clip_val_loader2 = DataLoader(clip_val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

In [7]:
print("=" * 50)
print("C: Text Features")
print("=" * 50)

model_c = CLIPWithTextFeatures(clip_model, clip_visual, num_classes=3)
res = fit(
    model_c, clip_train_loader, clip_val_loader2,
    name="yuda4_C_text_features",
    encoder_name="clip_vit_b32",
    epochs_head=5,
    epochs_finetune=10,
    lr_head=1e-3,
    lr_finetune=1e-4,
    patience=5,
    device=DEVICE,
)
print(f"C Macro F1: {res['best_val_f1']:.4f}")
results.append({'name': 'C_text_features', 'best_val_f1': res['best_val_f1'], 'f1_per_class': res['f1_per_class']})

/home/prayudahlah/projects/external/big-data-big-trouble/notebooks/experiments/../../modules/training/train.py:86: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


C: Text Features

=== yuda4_C_text_features: Phase 1 — Head Only ===


yuda4_C_text_features train:   0%|          | 0/664 [00:00<?, ?it/s]/home/prayudahlah/projects/external/big-data-big-trouble/notebooks/experiments/../../modules/training/train.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
yuda4_C_text_features val:   0%|          | 0/166 [00:00<?, ?it/s]                          /home/prayudahlah/projects/external/big-data-big-trouble/notebooks/experiments/../../modules/training/train.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


  E01: train_loss=0.1476  val_f1=0.9715  best=0.9715


yuda4_C_text_features train:   0%|          | 0/664 [00:00<?, ?it/s]/home/prayudahlah/projects/external/big-data-big-trouble/notebooks/experiments/../../modules/training/train.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
yuda4_C_text_features val:   0%|          | 0/166 [00:00<?, ?it/s]                          /home/prayudahlah/projects/external/big-data-big-trouble/notebooks/experiments/../../modules/training/train.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


...

  E09: train_loss=0.4381  val_f1=0.7060  best=0.9803


yuda4_C_text_features train:   0%|          | 0/664 [00:00<?, ?it/s]/home/prayudahlah/projects/external/big-data-big-trouble/notebooks/experiments/../../modules/training/train.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
yuda4_C_text_features val:   0%|          | 0/166 [00:00<?, ?it/s]                         /home/prayudahlah/projects/external/big-data-big-trouble/notebooks/experiments/../../modules/training/train.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


  E10: train_loss=0.3484  val_f1=0.7385  best=0.9803
  Early stopping at epoch 10


yuda4_C_text_features val:   0%|          | 0/166 [00:00<?, ?it/s]/home/prayudahlah/projects/external/big-data-big-trouble/notebooks/experiments/../../modules/training/train.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():


C Macro F1: 0.9803


## D: Multi-Encoder (CLIP + ConvNeXt)
Concat kedua features → classifier. Free keduanya lalu fine-tune.

In [8]:
class DualTransformDataset(Dataset):
    def __init__(self, df, transform_a, transform_b):
        self.df = df
        self.transform_a = transform_a
        self.transform_b = transform_b
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row['path']).convert('RGB')
        return self.transform_a(img), self.transform_b(img), _CLASS_TO_IDX[row['label']]

dual_train_ds = DualTransformDataset(df_train, clip_transform, conv_train_tfm)
dual_val_ds = DualTransformDataset(df_val, clip_transform, conv_val_tfm)
dual_train_loader = DataLoader(dual_train_ds, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)
dual_val_loader = DataLoader(dual_val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

In [9]:
# Reload CLIP visual
clip_visual = clip_model.visual.to(DEVICE)
clip_dim = clip_visual.output_dim

class MultiEncoderClassifier(nn.Module):
    def __init__(self, clip_visual, clip_dim, convnext_name='convnext_tiny', num_classes=3):
        super().__init__()
        self.clip_encoder = clip_visual
        self.conv_encoder = timm.create_model(convnext_name, pretrained=True, num_classes=0)
        with torch.no_grad():
            dummy = torch.randn(1, 3, 224, 224)
            self.conv_dim = self.conv_encoder(dummy).shape[-1]
        total_dim = clip_dim + self.conv_dim
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(total_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )
    def forward(self, clip_x, conv_x):
        clip_feat = self.clip_encoder(clip_x)
        conv_feat = self.conv_encoder(conv_x)
        combined = torch.cat([clip_feat, conv_feat], dim=1)
        return self.classifier(combined)
    def freeze_encoder(self):
        for p in self.clip_encoder.parameters():
            p.requires_grad = False
        for p in self.conv_encoder.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.clip_encoder.parameters():
            p.requires_grad = True
        for p in self.conv_encoder.parameters():
            p.requires_grad = True


def train_dual(model, train_loader, val_loader, name, epochs_head=5, epochs_finetune=10):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    best_f1 = 0.0
    best_state = None

    print(f"  Phase 1 — Head ({epochs_head} epoch)")
    model.freeze_encoder()
    opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
    for epoch in range(epochs_head):
        model.train()
        for cx, nx, y in tqdm(train_loader, desc=f"  Head E{epoch+1}", leave=False):
            cx, nx, y = cx.to(DEVICE), nx.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(cx, nx), y)
            loss.backward()
            opt.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for cx, nx, y in val_loader:
                cx, nx, y = cx.to(DEVICE), nx.to(DEVICE), y.to(DEVICE)
                preds.extend(model(cx, nx).argmax(dim=1).cpu().numpy())
                labs.extend(y.cpu().numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1;
            best_state = model.state_dict()
        print(f"    E{epoch+1}: val_f1={f1:.4f} (best={best_f1:.4f})")

    print(f"  Phase 2 — Full ({epochs_finetune} epoch)")
    model.unfreeze_encoder()
    model.load_state_dict(best_state)
    opt = torch.optim.AdamW([
        {'params': model.clip_encoder.parameters(), 'lr': 1e-4},
        {'params': model.conv_encoder.parameters(), 'lr': 1e-4},
        {'params': model.classifier.parameters(), 'lr': 1e-3},
    ], weight_decay=1e-4)
    for epoch in range(epochs_finetune):
        model.train()
        for cx, nx, y in tqdm(train_loader, desc=f"  Full E{epoch+1}", leave=False):
            cx, nx, y = cx.to(DEVICE), nx.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(cx, nx), y)
            loss.backward()
            opt.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for cx, nx, y in val_loader:
                cx, nx, y = cx.to(DEVICE), nx.to(DEVICE), y.to(DEVICE)
                preds.extend(model(cx, nx).argmax(dim=1).cpu().numpy())
                labs.extend(y.cpu().numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1;
            best_state = model.state_dict()
        print(f"    E{epoch+1}: val_f1={f1:.4f} (best={best_f1:.4f})")

    model.load_state_dict(best_state)
    result = {
        'name': name,
        'best_val_f1': best_f1,
        'f1_per_class': [],
    }
    torch.save(best_state, RESULTS / f'{name}.pt')
    with open(RESULTS / f'{name}.json', 'w') as f:
        import json
        json.dump(result, f, indent=2)
    return result

In [10]:
print("=" * 50)
print("D: Multi-Encoder")
print("=" * 50)

model_d = MultiEncoderClassifier(clip_visual, clip_dim, convnext_name='convnext_tiny')
res = train_dual(model_d, dual_train_loader, dual_val_loader, 'yuda4_D_multi_encoder', epochs_head=5, epochs_finetune=10)
print(f"D Macro F1: {res['best_val_f1']:.4f}")
results.append({'name': 'D_multi_encoder', 'best_val_f1': res['best_val_f1'], 'f1_per_class': []})

D: Multi-Encoder
  Phase 1 — Head (5 epoch)


    E1: val_f1=0.9418 (best=0.9418)


    E2: val_f1=0.9520 (best=0.9520)


    E3: val_f1=0.9486 (best=0.9520)


    E4: val_f1=0.9473 (best=0.9520)


...

    E6: val_f1=0.9352 (best=0.9554)


    E7: val_f1=0.9364 (best=0.9554)


    E8: val_f1=0.9307 (best=0.9554)


    E9: val_f1=0.9104 (best=0.9554)


    E10: val_f1=0.9369 (best=0.9554)
D Macro F1: 0.9554


---
## Summary

In [11]:
summary = pd.DataFrame(results)
summary = summary.sort_values('best_val_f1', ascending=False)
summary.to_csv(RESULTS / 'yuda4_summary.csv', index=False)
summary

,name,best_val_f1,f1_per_class
4,C_text_features,0.980287,"[0.7582774291278944, 0.6108979278587874, 0.846..."
3,B_conf_filter_0.99,0.957092,"[0.9447388342165026, 0.9623224212476837, 0.964..."
5,D_multi_encoder,0.955420,[]
2,B_conf_filter_0.95,0.955079,"[0.9426686960933536, 0.9582822085889571, 0.964..."
1,B_conf_filter_0.9,0.949061,"[0.935270132517839, 0.950519242516799, 0.96139..."
0,A_zero_shot,0.851980,"[0.7995735607675906, 0.8650793650793651, 0.891..."
